In [0]:

# TSI - Diagnóstico de ingesta
# Solo lectura: no modifica archivos ni tablas

from pyspark.sql import functions as F

RUTA = "/Volumes/workspace/tsi/datasets/"

try:
    archivos = dbutils.fs.ls(RUTA)

    print(f"Ruta disponible: {RUTA}")
    print(f"Elementos encontrados: {len(archivos)}")

    for archivo in archivos:
        print(
            f"Nombre: {archivo.name} | "
            f"Tamaño: {archivo.size:,} bytes | "
            f"Directorio: {archivo.isDir()}"
        )

except Exception as error:
    print(f"No se pudo inspeccionar la ruta: {error}")
    raise


In [0]:

# Revisar disponibilidad de las tablas principales

tablas = [
    "workspace.tsi.international_observations",
    "workspace.tsi.international_modeling",
    "workspace.tsi.pems_bay_predicciones_v2",
    "workspace.tsi.pems_bay_monitoreo_v2"
]

for tabla in tablas:
    try:
        spark.table(tabla).limit(1).collect()
        print(f"OK: {tabla}")
    except Exception as error:
        print(f"ERROR: {tabla} - {error}")


In [0]:

# TSI - Inventario de archivos disponibles
# Solo lectura

from datetime import datetime, timezone

RUTA = "/Volumes/workspace/tsi/datasets/"

archivos = dbutils.fs.ls(RUTA)

inventario = []

for archivo in archivos:
    if archivo.isDir():
        continue

    fecha = (
        datetime.fromtimestamp(
            archivo.modificationTime / 1000,
            tz=timezone.utc
        ).isoformat()
        if archivo.modificationTime
        else None
    )

    inventario.append({
        "archivo": archivo.name,
        "ruta": archivo.path,
        "tamano_bytes": archivo.size,
        "ultima_modificacion_utc": fecha
    })

display(spark.createDataFrame(inventario))


In [0]:

# TSI - Identificación de archivos mediante SHA-256
# Solo lectura. No modifica archivos ni tablas.

import hashlib
from pathlib import Path

RUTA = Path("/Volumes/workspace/tsi/datasets")

inventario_hash = []

for archivo in RUTA.glob("*.parquet"):
    sha256 = hashlib.sha256()

    with archivo.open("rb") as f:
        while True:
            bloque = f.read(4 * 1024 * 1024)

            if not bloque:
                break

            sha256.update(bloque)

    inventario_hash.append({
        "archivo": archivo.name,
        "tamano_bytes": archivo.stat().st_size,
        "sha256": sha256.hexdigest()
    })

display(spark.createDataFrame(inventario_hash))


In [0]:

# TSI - Crear tabla de control de ingesta
# No modifica las tablas históricas del proyecto.

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tsi.control_ingesta_archivos (
    archivo STRING,
    ruta STRING,
    sha256 STRING,
    tamano_bytes BIGINT,
    fecha_registro TIMESTAMP,
    estado STRING
)
USING DELTA
""")

print("Tabla de control disponible.")


In [0]:

# TSI - Registrar archivos conocidos sin duplicar hashes
# Estado: referencia_historica
# NO significa que el archivo haya sido procesado por este control.

from datetime import datetime, timezone
from pyspark.sql import Row

registros = []

for item in inventario_hash:
    registros.append(Row(
        archivo=item["archivo"],
        ruta=f"/Volumes/workspace/tsi/datasets/{item['archivo']}",
        sha256=item["sha256"],
        tamano_bytes=int(item["tamano_bytes"]),
        fecha_registro=datetime.now(timezone.utc),
        estado="referencia_historica"
    ))

df_nuevos = spark.createDataFrame(registros)
df_nuevos.createOrReplaceTempView("tsi_archivos_detectados")

spark.sql("""
MERGE INTO workspace.tsi.control_ingesta_archivos AS destino
USING tsi_archivos_detectados AS origen
ON destino.sha256 = origen.sha256
WHEN NOT MATCHED THEN
  INSERT (
    archivo, ruta, sha256, tamano_bytes,
    fecha_registro, estado
  )
  VALUES (
    origen.archivo, origen.ruta, origen.sha256,
    origen.tamano_bytes, origen.fecha_registro,
    origen.estado
  )
""")

display(
    spark.table("workspace.tsi.control_ingesta_archivos")
)


In [0]:

# TSI - Detección de archivos nuevos
# Solo lectura: no modifica archivos ni tablas.

import hashlib
from pathlib import Path

RUTA = Path("/Volumes/workspace/tsi/datasets")

# Consultar hashes registrados
hashes_conocidos = {
    fila["sha256"]
    for fila in spark.table(
        "workspace.tsi.control_ingesta_archivos"
    ).select("sha256").collect()
}

resultados = []

for archivo in sorted(RUTA.glob("*.parquet")):
    sha256 = hashlib.sha256()

    with archivo.open("rb") as f:
        for bloque in iter(lambda: f.read(4 * 1024 * 1024), b""):
            sha256.update(bloque)

    hash_actual = sha256.hexdigest()

    estado = (
        "CONOCIDO"
        if hash_actual in hashes_conocidos
        else "PENDIENTE_REVISION"
    )

    resultados.append({
        "archivo": archivo.name,
        "sha256": hash_actual,
        "estado": estado
    })

display(spark.createDataFrame(resultados))

pendientes = sum(
    item["estado"] == "PENDIENTE_REVISION"
    for item in resultados
)

print(f"Archivos encontrados: {len(resultados)}")
print(f"Archivos pendientes de revisión: {pendientes}")
